# Refactorización y Code Smells

## Introducción
Un *code smell* es un síntoma de diseño fácil de detectar en el código
fuente: no es un error en sí mismo (el programa puede funcionar
perfectamente), pero suele delatar un problema más profundo de diseño.
La **refactorización** es el proceso sistemático de mejorar el diseño
interno del código **sin cambiar su comportamiento externo** — la
herramienta principal para pasar de código sucio a código limpio y,
cuando el smell revela una falla estructural, el paso previo a aplicar
un patrón de diseño.

Este notebook recorre los 13 code smells más comunes cubiertos en el
módulo de clase, organizados en las 5 categorías estándar
(clasificación de refactoring.guru): **Bloaters**, **Object-Orientation
Abusers**, **Change Preventers**, **Dispensables** y **Couplers**. Por
cada uno verás una versión "con el smell", su versión refactorizada y
una explicación de la técnica aplicada.

## Objetivos
- Reconocer visualmente los code smells más frecuentes en código Python.
- Aplicar la técnica de refactorización correspondiente a cada smell.
- Conectar cada smell con la técnica de refactor que lo resuelve, como
  paso de diagnóstico previo a decidir un patrón de diseño.
- Practicar el mismo tipo de análisis que se espera en el diagnóstico
  del "antes" de la Actividad 4 (proyecto integrador).

# Bloaters — código que se hinchó

No aparecen de golpe: se acumulan gradualmente a medida que el software evoluciona.

## Serie: Refactorización y Code Smells

Este contenido está dividido en 6 notebooks — uno por categoría de code
smell (clasificación de refactoring.guru) más un cierre de ejercicios:

1. **`01_bloaters.ipynb`** — Bloaters
2. `02_object_orientation_abusers.ipynb` — Object-Orientation Abusers
3. `03_change_preventers.ipynb` — Change Preventers
4. `04_dispensables.ipynb` — Dispensables
5. `05_couplers.ipynb` — Couplers
6. `06_ejercicios_autoevaluacion.ipynb` — Ejercicios y autoevaluación


### 1. Long Method (método largo)

**Definición:** Un método que ha crecido demasiado y asume varias responsabilidades a la vez (validar, calcular, imprimir...).

**Síntoma:** Es psicológicamente más barato añadir 2 líneas a un método existente que crear uno nuevo — el método se vuelve un "Hotel California": entra código, pero nada sale.

**Técnica de refactor:** Extract Method

#### Con el smell

In [ ]:
class Item:
    def __init__(self, nombre, precio, cantidad):
        self.nombre = nombre
        self.precio = precio
        self.cantidad = cantidad

class Pedido:
    def __init__(self, id, items):
        self.id = id
        self.items = items

def generar_reporte(pedido):
    print(f"Reporte de pedido #{pedido.id}")
    total = 0
    for item in pedido.items:
        subtotal = item.precio * item.cantidad
        total += subtotal
        print(f"  {item.nombre}: {subtotal:.2f}")
    if total > 100:
        total *= 0.9  # descuento por volumen
    print(f"TOTAL: {total:.2f}")
    return total

pedido = Pedido(1, [Item("Teclado", 80, 1), Item("Mouse", 30, 2)])
generar_reporte(pedido)

#### Refactorizado

In [ ]:
def _calcular_total(pedido):
    total = sum(item.precio * item.cantidad for item in pedido.items)
    return total * 0.9 if total > 100 else total

def _imprimir_detalle(pedido, total):
    print(f"Reporte de pedido #{pedido.id}")
    for item in pedido.items:
        print(f"  {item.nombre}: {item.precio * item.cantidad:.2f}")
    print(f"TOTAL: {total:.2f}")

def generar_reporte(pedido):
    total = _calcular_total(pedido)
    _imprimir_detalle(pedido, total)
    return total

pedido = Pedido(1, [Item("Teclado", 80, 1), Item("Mouse", 30, 2)])
generar_reporte(pedido)

**Explicación:** `Extract Method` divide `generar_reporte` en dos métodos con una sola responsabilidad cada uno: `_calcular_total` (regla de negocio) e `_imprimir_detalle` (presentación). El resultado impreso es idéntico, pero cada pieza ahora se puede leer, probar y reutilizar por separado.

### 2. Large Class (clase grande)

**Definición:** Una clase que acumula demasiados campos, métodos y responsabilidades — viola SRP.

**Síntoma:** Cada requerimiento nuevo se resuelve agregándole "algo más" a la misma clase, en vez de crear una clase nueva.

**Técnica de refactor:** Extract Class

#### Con el smell

In [ ]:
class Empleado:
    def __init__(self, nombre, salario):
        self.nombre = nombre
        self.salario = salario

    def calcular_impuesto(self):
        return self.salario * 0.19

    def calcular_bono(self, meses):
        return self.salario * 0.1 * meses

    def imprimir_recibo(self):
        imp = self.calcular_impuesto()
        neto = self.salario - imp
        print(f"{self.nombre}: neto {neto:.2f}")

empleado = Empleado("Ana", 3_000_000)
empleado.imprimir_recibo()

#### Refactorizado

In [ ]:
class Empleado:
    def __init__(self, nombre, salario):
        self.nombre = nombre
        self.salario = salario

class CalculadoraNomina:
    def impuesto(self, empleado):
        return empleado.salario * 0.19

    def bono(self, empleado, meses):
        return empleado.salario * 0.1 * meses

class ReciboImpresora:
    def imprimir(self, empleado, calculadora):
        imp = calculadora.impuesto(empleado)
        neto = empleado.salario - imp
        print(f"{empleado.nombre}: neto {neto:.2f}")

empleado = Empleado("Ana", 3_000_000)
calc = CalculadoraNomina()
ReciboImpresora().imprimir(empleado, calc)

**Explicación:** `Extract Class` separa datos (`Empleado`), reglas de nómina (`CalculadoraNomina`) e impresión (`ReciboImpresora`) en tres clases con una sola razón para cambiar cada una.

### 3. Primitive Obsession (obsesión primitiva)

**Definición:** Usar tipos primitivos (str, int, listas) para representar conceptos de dominio (teléfono, dinero, dirección) en vez de una clase dedicada.

**Síntoma:** "Es solo un campo simple para guardar un dato" — hasta que la validación y el formateo de ese dato se repiten en cada función que lo usa.

**Técnica de refactor:** Replace Data Value with Object

#### Con el smell

In [ ]:
class Cliente:
    def __init__(self, nombre, telefono):
        self.nombre = nombre
        self.telefono = telefono  # str sin validar

def enviar_sms(telefono):
    limpio = telefono.replace("-", "")
    if len(limpio) != 10:
        raise ValueError("Teléfono inválido")
    print(f"SMS enviado a {limpio}")

cliente = Cliente("Ana", "300-555-1234")
enviar_sms(cliente.telefono)

#### Refactorizado

In [ ]:
class Telefono:
    def __init__(self, numero):
        limpio = numero.replace("-", "")
        if len(limpio) != 10:
            raise ValueError("Teléfono inválido")
        self._numero = limpio

    def enviar_sms(self):
        print(f"SMS enviado a {self._numero}")

class Cliente:
    def __init__(self, nombre, telefono):
        self.nombre = nombre
        self.telefono = Telefono(telefono)

cliente = Cliente("Ana", "300-555-1234")
cliente.telefono.enviar_sms()

**Explicación:** `Replace Data Value with Object` crea `Telefono` como clase dedicada: la validación vive en un solo lugar, y cualquier otra parte del sistema que necesite enviar SMS reutiliza el mismo método en vez de reimplementar la limpieza del número.

### 4. Long Parameter List (lista larga de parámetros)

**Definición:** Un método o función que exige demasiados parámetros (más de 3-4), muchos de ellos relacionados entre sí.

**Síntoma:** Cada llamada se vuelve difícil de leer y de armar sin errores: es fácil pasar dos parámetros del mismo tipo en el orden equivocado.

**Técnica de refactor:** Introduce Parameter Object

#### Con el smell

In [ ]:
def crear_reserva(nombre, email, telefono, fecha_entrada, fecha_salida, num_huespedes, con_desayuno):
    print(f"Reserva para {nombre} ({email}, {telefono})")
    print(f"{fecha_entrada} -> {fecha_salida}, {num_huespedes} huésped(es)")
    print(f"Desayuno incluido: {con_desayuno}")

crear_reserva("Ana", "ana@mail.com", "3001234567", "2026-09-01", "2026-09-05", 2, True)

#### Refactorizado

In [ ]:
class Huesped:
    def __init__(self, nombre, email, telefono):
        self.nombre = nombre
        self.email = email
        self.telefono = telefono

class Estadia:
    def __init__(self, fecha_entrada, fecha_salida, num_huespedes, con_desayuno):
        self.fecha_entrada = fecha_entrada
        self.fecha_salida = fecha_salida
        self.num_huespedes = num_huespedes
        self.con_desayuno = con_desayuno

def crear_reserva(huesped, estadia):
    print(f"Reserva para {huesped.nombre} ({huesped.email}, {huesped.telefono})")
    print(f"{estadia.fecha_entrada} -> {estadia.fecha_salida}, {estadia.num_huespedes} huésped(es)")
    print(f"Desayuno incluido: {estadia.con_desayuno}")

huesped = Huesped("Ana", "ana@mail.com", "3001234567")
estadia = Estadia("2026-09-01", "2026-09-05", 2, True)
crear_reserva(huesped, estadia)

**Explicación:** `Introduce Parameter Object` agrupa los parámetros que siempre viajan juntos (`Huesped`, `Estadia`) en objetos con nombre. La llamada queda más corta y el orden de los datos ya no depende de recordar la posición exacta de 7 argumentos.

### 5. Data Clumps (grupos de datos)

**Definición:** Un grupo de variables (parámetros o atributos) que siempre aparecen juntas en distintas partes del código y en realidad representan un mismo concepto.

**Síntoma:** El mismo par o trío de variables (p. ej. x, y) se repite como parámetros en varias funciones distintas.

**Técnica de refactor:** Extract Class

#### Con el smell

In [ ]:
def distancia(x1, y1, x2, y2):
    return ((x2 - x1) ** 2 + (y2 - y1) ** 2) ** 0.5

def mover_punto(x, y, dx, dy):
    return x + dx, y + dy

def imprimir_punto(x, y):
    print(f"({x}, {y})")

imprimir_punto(*mover_punto(0, 0, 3, 4))

#### Refactorizado

In [ ]:
class Punto:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def distancia(self, otro):
        return ((otro.x - self.x) ** 2 + (otro.y - self.y) ** 2) ** 0.5

    def mover(self, dx, dy):
        return Punto(self.x + dx, self.y + dy)

    def __str__(self):
        return f"({self.x}, {self.y})"

p = Punto(0, 0)
print(p.mover(3, 4))

**Explicación:** `Extract Class` convierte el par `x, y` (que siempre viajaba junto) en una clase `Punto` con sus propias operaciones (`distancia`, `mover`). El concepto ahora vive en un solo lugar, en vez de repetirse como pareja de parámetros.